# Transcributor — GPU Backend (WhisperX)

Этот ноутбук запускает API сервер в Google Colab и связывает его с вашим веб-приложением через ngrok.

In [ ]:
# @title 1. Настройка параметров

NGROK_AUTH_TOKEN = "YOUR_NGROK_TOKEN" # @param {type:"string"}
NGROK_STATIC_DOMAIN = "YOUR_NGROK_DOMAIN" # @param {type:"string"}
COLAB_SECRET = "my-secret-key-123" # @param {type:"string"}
HF_TOKEN = "YOUR_HF_TOKEN" # @param {type:"string"}


In [ ]:
# @title 2. Установка зависимостей (займёт ~2-3 минуты)
!pip install whisperx flask pyngrok httpx nest_asyncio
!apt install libcudnn8 libcudnn8-dev -y


In [ ]:
# @title 3. Загрузка модели WhisperX
import whisperx
import gc

device = "cuda"
compute_type = "float16" # или int8 для экономии памяти
model_name = "large-v3-turbo"

print(f"Загрузка модели {model_name}...")
model = whisperx.load_model(model_name, device, compute_type=compute_type)
print("Модель успешно загружена!")


In [ ]:
# @title 4. Запуск API Сервера
from flask import Flask, request, jsonify
from pyngrok import ngrok, conf
import nest_asyncio
import os

app = Flask(__name__)

@app.route('/', methods=['GET'])
def health():
    return jsonify({"status": "ok"})

@app.route('/transcribe', methods=['POST'])
def api_transcribe():
    secret = request.headers.get("Authorization", "").replace("Bearer ", "")
    if secret != COLAB_SECRET:
        return jsonify({"error": "Unauthorized"}), 401
        
    if 'audio' not in request.files:
        return jsonify({"error": "No audio file provided"}), 400
        
    file = request.files['audio']
    audio_path = "/tmp/input_audio.mp3"
    file.save(audio_path)
    
    try:
        print("Начало транскрибации...")
        audio = whisperx.load_audio(audio_path)
        result = model.transcribe(audio, batch_size=8)
        
        # Опциональная диаризация (раскомментируй, если нужна)
        # if HF_TOKEN and HF_TOKEN != "YOUR_HF_TOKEN":
        #     print("Выполняется диаризация...")
        #     diarize_model = whisperx.DiarizationPipeline(use_auth_token=HF_TOKEN, device=device)
        #     diarize_segments = diarize_model(audio)
        #     result = whisperx.assign_word_speakers(diarize_segments, result)
        
        text = " ".join([s["text"].strip() for s in result["segments"]])
        print("Транскрибация завершена.")
        
        return jsonify({
            "text": text,
            "language": result["language"]
        })
    except Exception as e:
        print(f"Ошибка: {str(e)}")
        return jsonify({"error": str(e)}), 500
    finally:
        if os.path.exists(audio_path):
            os.remove(audio_path)

conf.get_default().auth_token = NGROK_AUTH_TOKEN
public_url = ngrok.connect(5000, "http", domain=NGROK_STATIC_DOMAIN).public_url
print(f"\n{'='*50}")
print(f"✅ СЕРВЕР ЗАПУЩЕН!")
print(f"🌍 Публичный URL: {public_url}")
print(f"{'='*50}\n")

# Позволяем Flask работать внутри Colab
nest_asyncio.apply()
app.run(port=5000)
